In [13]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

DATA_DIR = Path('/work/gr-fe/bryan/ECCB2026_TEST/data_tmp/TCGA-BRCA/')
target_col = 'paper_BRCA_Subtype_PAM50'

## Transcriptomics

In [2]:
modality = 'mRNA'

expr_tran = pd.read_csv(f'{data_dir}/datExpr_{modality}.csv', index_col=0).T
meta_tran = pd.read_csv(f'{data_dir}/datMeta_{modality}.csv', index_col=0).dropna()

common_pat = set(expr_tran.index) & set(meta_tran.index)

meta_tran = meta_tran.loc[list(common_pat)]
expr_tran = expr_tran.loc[list(common_pat)]

## Proteomics

In [3]:
modality = 'RPPA'

expr_prot = pd.read_csv(f'{data_dir}/datExpr_{modality}.csv', index_col=0)
meta_prot = pd.read_csv(f'{data_dir}/datMeta_{modality}.csv', index_col=0).dropna()

common_pat = set(expr_prot.index) & set(meta_prot.index)

meta_prot = meta_prot.loc[list(common_pat)]
expr_prot = expr_prot.loc[list(common_pat)]

## DNA Methylation

In [4]:
modality = 'DNAm'

expr_meth = pd.read_csv(f'{data_dir}/datExpr_{modality}.csv', index_col=0)
meta_meth = pd.read_csv(f'{data_dir}/datMeta_{modality}.csv', index_col=0).dropna()

common_pat = set(expr_meth.index) & set(meta_meth.index)

meta_meth = meta_meth.loc[list(common_pat)]
expr_meth = expr_meth.loc[list(common_pat)]

### Intersect to patients with all modalities, subset to 500 patients, cleaning checks

In [22]:
# 1) common patients across all expr+meta
common_patients = sorted(
    set(expr_tran.index) & set(meta_tran.index) &
    set(expr_prot.index) & set(meta_prot.index) &
    set(expr_meth.index) & set(meta_meth.index)
)
print(f"Common patients across all expr+meta: {len(common_patients)}")

# 2) subset + align to common patients
expr_tran = expr_tran.loc[common_patients]
meta_tran = meta_tran.loc[common_patients]

expr_prot = expr_prot.loc[common_patients]
meta_prot = meta_prot.loc[common_patients]

expr_meth = expr_meth.loc[common_patients]
meta_meth = meta_meth.loc[common_patients]

# 3) missingness checks
def check_no_missing(name, df):
    n_missing = int(df.isna().to_numpy().sum())
    if n_missing:
        raise ValueError(f"{name} has {n_missing} missing values (NaN/NA).")

check_no_missing("transcriptomics.expr", expr_tran)
check_no_missing("transcriptomics.meta", meta_tran)
check_no_missing("proteomics.expr", expr_prot)
check_no_missing("proteomics.meta", meta_prot)
check_no_missing("methylation.expr", expr_meth)
check_no_missing("methylation.meta", meta_meth)
print("Missingness check passed for all omics (expr + meta).")

# 4) pick 500 patients with as balanced classes as possible from a target column in meta_tran
n_total = 500
seed = 0
rng = np.random.default_rng(seed)

y = meta_tran[target_col].dropna()
available = pd.Index(common_patients).intersection(y.index)

y = y.loc[available]
counts = y.value_counts()
classes = counts.index.to_list()
n_classes = len(classes)

if n_classes == 0:
    raise ValueError(f"No non-missing values found in meta_tran['{target_col}'].")

q = n_total // n_classes  # base quota per class

selected = []
for c in classes:
    members = y.index[y == c].to_numpy()
    take = min(len(members), q)        # if class has fewer than q, take all
    selected.append(rng.choice(members, size=take, replace=False))

selected = np.concatenate(selected) if selected else np.array([], dtype=object)

# fill remaining slots, always taking from the class with the most remaining samples
remaining = n_total - len(selected)
remaining_by_class = {c: np.setdiff1d(y.index[y == c].to_numpy(), selected) for c in classes}

while remaining > 0:
    c = max(classes, key=lambda k: remaining_by_class[k].size)
    pool_c = remaining_by_class[c]
    if pool_c.size == 0:
        break
    pick = rng.choice(pool_c, size=1, replace=False)
    selected = np.concatenate([selected, pick])
    remaining_by_class[c] = np.setdiff1d(pool_c, pick)
    remaining -= 1

selected = pd.Index(selected).unique()

print(f"Selected patients: {len(selected)}")
print(meta_tran.loc[selected, target_col].value_counts())

# 5) subset all omics to the selected 500 patients
expr_tran = expr_tran.loc[selected]
meta_tran = meta_tran.loc[selected]

expr_prot = expr_prot.loc[selected]
meta_prot = meta_prot.loc[selected]

expr_meth = expr_meth.loc[selected]
meta_meth = meta_meth.loc[selected]

# 6) save back (overwrite)
pd.to_pickle({"expr": expr_tran, "meta": meta_tran}, DATA_DIR / "transcriptomics.pkl")
pd.to_pickle({"expr": expr_prot, "meta": meta_prot}, DATA_DIR / "proteomics.pkl")
pd.to_pickle({"expr": expr_meth, "meta": meta_meth}, DATA_DIR / "methylation.pkl")

print("Saved updated (common + balanced-500) pickles to:", DATA_DIR)

# 7) save into one pickle called 'omics.pkl'
omics_out = {
    "transcriptomics": expr_tran,
    "proteomics": expr_prot,
    "methylation": expr_meth,
    "meta" : meta_tran[target_col]
}

out_path = DATA_DIR / "omics.pkl"
pd.to_pickle(omics_out, out_path)
print("Saved:", out_path)


Common patients across all expr+meta: 500
Missingness check passed for all omics (expr + meta).
Selected patients: 500
paper_BRCA_Subtype_PAM50
LumA      237
LumB      100
Basal      97
Her2       41
Normal     25
Name: count, dtype: int64
Saved updated (common + balanced-500) pickles to: /work/gr-fe/bryan/ECCB2026_TEST/data_tmp/TCGA-BRCA
Saved: /work/gr-fe/bryan/ECCB2026_TEST/data_tmp/TCGA-BRCA/omics.pkl
